# 01. Data Acquisition

**Objective:** Retrieve Manhattan collision records from NYC Open Data for 2019–2025, save the seven required columns, and verify completeness, temporal coverage and coordinate quality. This notebook retains raw rows; Notebook 02 applies cleaning and saves the cleaned dataset.

## 1. Configuration

Use the official Motor Vehicle Collisions API (`h9gi-nx95`), the `MANHATTAN` borough label, and the date interval from 2019-01-01 inclusive to 2026-01-01 exclusive. Run from the `notebooks/` directory.

In [ ]:
from pathlib import Path
import pandas as pd
import requests

# Official NYC collision dataset API
API_URL = "https://data.cityofnewyork.us/resource/h9gi-nx95.json"

# Initial date range
START_DATE = "2019-01-01"
END_DATE = "2026-01-01"  # Exclusive: includes 2025

# Columns needed for RoadRisk
COLUMNS = [
    "collision_id",
    "crash_date",
    "crash_time",
    "borough",
    "latitude",
    "longitude",
    "on_street_name"
]

OUTPUT = "../data/raw/manhattan_collisions.csv"
BATCH_SIZE = 10000

base_filter = (
    f"crash_date >= '{START_DATE}' AND "
    f"crash_date < '{END_DATE}' AND "
    "borough = 'MANHATTAN'"
)

download_complete = False

## 2. Download Manhattan collisions

Download batches ordered by collision ID into a `.part` file. The existing raw
CSV is replaced only after the completed download passes section 3's verification.
Failed downloads leave the original CSV intact. For local checks, run configuration
and section 3 without rerunning the download.

In [ ]:
download_complete = False
Path(OUTPUT).parent.mkdir(parents=True, exist_ok=True)
temp_output = Path(OUTPUT + ".part")
temp_output.unlink(missing_ok=True)
last_id = None
total_rows = 0

while True:
    where = base_filter

    if last_id is not None:
        safe_id = last_id.replace("'", "''")
        where += f" AND collision_id > '{safe_id}'"

    params = {
        "$select": ", ".join(COLUMNS),
        "$where": where,
        "$order": "collision_id ASC",
        "$limit": BATCH_SIZE
    }

    response = requests.get(API_URL, params=params, timeout=120)

    if not response.ok:
        print("API error:", response.text[:1000])
        response.raise_for_status()

    records = response.json()

    if not records:
        break

    batch = pd.DataFrame(records).reindex(columns=COLUMNS)

    batch.to_csv(
        temp_output,
        mode="w" if total_rows == 0 else "a",
        header=(total_rows == 0),
        index=False
    )

    total_rows += len(batch)
    last_id = str(batch["collision_id"].iloc[-1])

    print(f"Downloaded {total_rows:,} records")

    if len(records) < BATCH_SIZE:
        break

if total_rows == 0:
    raise ValueError("No records downloaded")

download_complete = True

print(f"\nDownload complete: {total_rows:,} records")
print(f"Staged for verification: {temp_output}")

## 3. Verify completeness and coverage

Check the saved schema, row count, unique collision IDs, borough labels and date range. Confirm that every quarter in the requested interval is represented, and show annual counts once.

In [ ]:
df = pd.read_csv(temp_output if download_complete else OUTPUT)

assert df.columns.tolist() == COLUMNS
assert df["collision_id"].notna().all()
assert df["collision_id"].is_unique
assert df["borough"].eq("MANHATTAN").all()

# Convert dates once for the local coverage and quality checks.
audit = df.copy()
audit["crash_date"] = pd.to_datetime(audit["crash_date"], errors="coerce")
assert audit["crash_date"].notna().all()
assert audit["crash_date"].ge(pd.Timestamp(START_DATE)).all()
assert audit["crash_date"].lt(pd.Timestamp(END_DATE)).all()

quarter_counts = (
    audit.groupby(audit["crash_date"].dt.to_period("Q")).size()
)
expected_quarters = pd.period_range(
    pd.Timestamp(START_DATE).to_period("Q"),
    (pd.Timestamp(END_DATE) - pd.Timedelta(days=1)).to_period("Q"),
    freq="Q"
)
assert quarter_counts.index.equals(expected_quarters)

if download_complete:
    temp_output.replace(OUTPUT)
    download_complete = False

print("Saved dataset shape:", df.shape)
print("Earliest date:", audit["crash_date"].min().date())
print("Latest date:", audit["crash_date"].max().date())
print("Quarters covered:", len(quarter_counts))
print("Local schema and coverage checks passed.")

display(
    audit.groupby(audit["crash_date"].dt.year)
    .size()
    .rename("collisions")
    .to_frame()
)

### 3.1. Live API completeness check

Optional: compare the current API count with the local file. The source can change,
so this network check is separate from the local dataset verification above.

In [ ]:
# Compare the saved download with the current API count.
response = requests.get(
    API_URL,
    params={
        "$select": "count(*) as total",
        "$where": base_filter
    },
    timeout=30
)
response.raise_for_status()

api_total = int(response.json()[0]["total"])
local_total = len(df)

print("API matching records:", api_total)
print("Downloaded records:", local_total)
print("Difference:", api_total - local_total)

## 4. Data-quality audit

Count missing coordinates and nonmissing coordinates outside the original broad NYC audit bounds: latitude 40.5–41.0 and longitude −74.3 to −73.6. These checks retain every raw row. Geographic cleaning and filtering belong to Notebook 02.

In [5]:
audit["latitude"] = pd.to_numeric(audit["latitude"], errors="coerce")
audit["longitude"] = pd.to_numeric(audit["longitude"], errors="coerce")

missing_coords = audit[["latitude", "longitude"]].isna().any(axis=1)

# Broad NYC bounds are an audit, not a Manhattan-boundary filter.
valid_coords = (
    audit["latitude"].between(40.5, 41.0)
    & audit["longitude"].between(-74.3, -73.6)
)
invalid_coords = ~valid_coords & ~missing_coords

print("Records with missing coordinates:", missing_coords.sum())
print(f"Missing-coordinate share: {missing_coords.mean():.2%}")
print("Invalid coordinates (excluding missing):", invalid_coords.sum())
print("Rows retained in the raw dataset:", len(df))


Records with missing coordinates: 2734
Missing-coordinate share: 2.76%
Invalid coordinates (excluding missing): 1029
Rows retained in the raw dataset: 99102


## 5. Summary

The saved raw dataset, `data/raw/manhattan_collisions.csv`, contains **99,102 rows × 7 columns**, spanning 2019-01-01 to 2025-12-31 across 28 quarters. Collision IDs are unique, all records are labeled Manhattan, and no dates are missing.

The raw data retains 2,734 rows with missing coordinates and 1,029 additional rows outside the broad NYC audit bounds. Notebook 02 applies the cleaning rules and saves `data/processed/manhattan_collision_cleaned.csv`. The download and live API completeness check still require execution when refreshing the source data.